# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [32]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [33]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [34]:
spaceship=spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].astype(str).str[0]
spaceship = spaceship.drop(["PassengerId", "Name"], axis=1)
spaceship = pd.get_dummies(spaceship, drop_first=True)
spaceship

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,False,True,False,False,False,False,False,False,False,False,False,False,False,True
8689,18.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True,False,False,False,False,False,True,False,True,False,False
8690,26.0,0.0,0.0,1872.0,1.0,0.0,True,False,False,False,False,False,False,False,False,True,False,False,True,False
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,False,True,False,False,False,False,False,True,False,False,False,False,False,False


**Perform Train Test Split**

In [35]:
target = spaceship["Transported"]
features = spaceship.drop("Transported", axis=1)
X_train, X_test, y_train, y_test = train_test_split(features, target,test_size=0.2,random_state=42,shuffle=True)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [36]:
# Bagging
bagging = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    n_estimators=50,
    bootstrap=True,
    random_state=42
)

bagging.fit(X_train, y_train)
preds_bagging = bagging.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds_bagging))
print(classification_report(y_test, preds_bagging))


Accuracy: 0.7957639939485628
              precision    recall  f1-score   support

       False       0.79      0.79      0.79       653
        True       0.80      0.80      0.80       669

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



In [37]:
# Pasting
pasting = BaggingClassifier(
    estimator=KNeighborsClassifier(),
    n_estimators=50,
    bootstrap=False,
    random_state=42
)

pasting.fit(X_train, y_train)
preds_pasting = pasting.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds_pasting))
print(classification_report(y_test, preds_pasting))

Accuracy: 0.791981845688351
              precision    recall  f1-score   support

       False       0.79      0.80      0.79       653
        True       0.80      0.79      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



- Random Forests

In [38]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds_rf))
print(classification_report(y_test, preds_rf))

Accuracy: 0.8139183055975794
              precision    recall  f1-score   support

       False       0.81      0.81      0.81       653
        True       0.82      0.81      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Gradient Boosting

In [39]:
gb = GradientBoostingClassifier(random_state=42)

gb.fit(X_train, y_train)
preds_gb = gb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds_gb))
print(classification_report(y_test, preds_gb))

Accuracy: 0.8071104387291982
              precision    recall  f1-score   support

       False       0.84      0.76      0.79       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



- Adaptive Boosting

In [40]:
ada = AdaBoostClassifier(
    n_estimators=200,
    random_state=42
)

ada.fit(X_train, y_train)
preds_ada = ada.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds_ada))
print(classification_report(y_test, preds_ada))

Accuracy: 0.7874432677760969
              precision    recall  f1-score   support

       False       0.78      0.80      0.79       653
        True       0.80      0.78      0.79       669

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



Which model is the best and why?

In [41]:
print("Bagging:", accuracy_score(y_test, preds_bagging))
print("Pasting:", accuracy_score(y_test, preds_pasting))
print("Random Forest:", accuracy_score(y_test, preds_rf))
print("Gradient Boosting:", accuracy_score(y_test, preds_gb))
print("AdaBoost:", accuracy_score(y_test, preds_ada))

Bagging: 0.7957639939485628
Pasting: 0.791981845688351
Random Forest: 0.8139183055975794
Gradient Boosting: 0.8071104387291982
AdaBoost: 0.7874432677760969


In [42]:
# El mejor es Random Forest (0.81 de acc) seguido de Gradient Boosting. 
# Es normal, ya que estos modelos usan muchos árboles juntos 
# y pueden aprender patrones más complejos sin cometer demasiado overfitting.
